Data Importer and Filter Notebook

In [9]:
import pandas as pd
import yfinance as yf
import random
from collections import defaultdict
from pathlib import Path
from datetime import datetime, timedelta
from abc import ABC, abstractmethod


# Set project folder as parent file; allow for data file to be pulled from data folder in notebook (more robust access method)
project_root = Path.cwd().parent
data_path = project_root / "data" / "AssetDB.csv"

# Background reference; general properties to be used throughout the code, update as needed

eu_exchanges = {"Xetra","London Stock Exchange","Deutsche Boerse AG","Euronext Amsterdam", "NASDAQ Helsinki","SIX Swiss Exchange","Euronext Paris","Borsa Italiana","Bolsas y Mercados Espanoles","Oslo Bors Asa"}
us_exchanges = {"NYSE Arca","New York Stock Exchange","Nasdaq Stock Market"}

class Asset:

    n_assets = 0

    def __init__(self, ticker, name, asset_type, sector, exchange, currency, description):
        self.ticker = ticker
        self.name = name
        self.asset_type = asset_type
        self.sector = sector
        self.exchange = exchange
        self.currency = currency
        self.description = description
        self.region = "EU" if self.exchange in eu_exchanges else "US" if self.exchange in us_exchanges else "Unknown"
        Asset.n_assets += 1

def load_companies():
    df = pd.read_csv(data_path)
    companies = {row.stock:Asset(row.yfinance_ticker, row.name,row.asset_type, row.sector, row.exchange, row.currency, row.description) for row in df.itertuples()}
    return(companies,df)

companies, df = load_companies()

# Assets grouped by sector, kept for sector filtering later
sectors = defaultdict(list)
for a in companies.values():
    sectors[a.sector].append(a)

print(pd.Series([a.region for a in companies.values()]).value_counts())

US    28
EU    26
Name: count, dtype: int64


In [3]:
today = datetime.today()
startdate = today - timedelta(days = 50)
tickerlist = [a.ticker for a in companies.values()]
assetdata = yf.download(tickerlist, start = startdate, end = today)

[*********************100%***********************]  53 of 54 completed

1 Failed download:
['MSFT']: OperationalError('database is locked')


In [6]:
# General "Portfolio" abstract class as blueprint for manual or automatic portfolio creation
class Portfolio(ABC):
    def __init__(self, name):
        self.name = name
        self.portfolio_assets=[]

    @abstractmethod
    def build(self):
        pass

# Manual Portfolio subclass to be filled in with a manual list of tickers
class Manual(Portfolio):
    def __init__(self, name):
        super().__init__(name)

    def build(self, assetlist):
        for asset in assetlist:
            if asset not in self.portfolio_assets:
                self.portfolio_assets.append(asset)

# Automated Portfolio subclass to be filled with region and/or sector weightings (or neither for random selection)
# Asset selection is still randomized for any type of portfolio (Random/Region Split/Region + Sector Split); worth considering separating this code
# so the portfolios don't have to be created on each run and can be stored separately from the main data import and data analysis.
class Automated(Portfolio):
    def __init__(self, name, weights=None):
        super().__init__(name)
        self.weights = weights

    def build(self, n = None):
        if self.weights is None:
            self.portfolio_assets = random.sample(list(companies.values()), n)
            return

        for region, split in self.weights.items():
            if isinstance(split,dict): # Region + Sector splits
                for sector, pct in split.items():
                    pool = [a for a in companies.values() if a.region == region and a.sector == sector]
                    count = round(n * pct)
                    self.portfolio_assets += random.sample(pool, min(count, len(pool)))
            else: # Region split only
                pool = [a for a in companies.values() if a.region == region]
                count = round(n * split)
                self.portfolio_assets += random.sample(pool, min(count, len(pool)))

Portfolio1 = Automated("Randomized Portfolio")
Portfolio1.build(40)

Portfolio2 = Automated("60/40", weights={"US":0.6,"EU":0.4})
Portfolio2.build(30)

#for a in Portfolio2.portfolio_assets:
 #   print(str(a.ticker))


In [7]:
pd.read_csv("../data/AssetDB.csv")["exchange"].unique()


<StringArray>
[                  'NYSE Arca',                       'Xetra',
       'London Stock Exchange',         'Nasdaq Stock Market',
     'New York Stock Exchange',          'Deutsche Boerse AG',
          'Euronext Amsterdam',             'NASDAQ Helsinki',
          'SIX Swiss Exchange',              'Euronext Paris',
              'Borsa Italiana', 'Bolsas y Mercados Espanoles',
               'Oslo Bors Asa']
Length: 13, dtype: str